# 2 · The three operators of HDC

In notebook 1 we stored the sales orders as a graph and asked it exact questions. In this
notebook we build a different representation of the same nodes and relationship types, using
**hyperdimensional computing (HDC)**.

Why bother making a new representation? As we saw earlier, the graph needs an exact pattern to match.
HDC can turn a fuzzy or partial pattern into a *hypervector* (a wide vector with thousands of dimensions),
in such a way that **similar patterns get similar hypervectors**. Then, a query like "_find past
orders like this one_" becomes a similarity search, which a vector store like LanceDB is built for.
This notebook covers the building blocks of the HDC operators we need.

## A few terms first

HDC has its own vocabulary. Here's what each term means, in one line each. Each one gets its
own section below.

| Term | What it means |
|---|---|
| **hypervector** | A very long list of numbers (here 10,000 of them) that stands for one thing, such as a node or a relationship type. The meaning is spread across all the entries, so no single entry means anything by itself. |
| **codebook** | A lookup table that assigns each graph node (e.g. `east-london`) and relationship type (e.g. `SHIPS_FROM`) a random hypervector, reused whenever that node or type appears. |
| **similarity** | How alike two hypervectors are, measured with cosine similarity: 1 for identical, 0 for unrelated. |
| **binding ⊗** | Combines a key with its value, such as `SHIPS_FROM` with `east-london`. The result is unlike both inputs. |
| **bundling ⊕** | Combines several hypervectors into one that stays similar to each of them, like a set. |
| **permutation ρ** | Shuffles a hypervector's entries in a fixed way, to mark order or position, such as which end of an edge is which. |
| **clean-up** | Turning a noisy hypervector back into a name, by finding its most similar entry in the codebook. |

There's no training anywhere in this notebook. Everything is random numbers and arithmetic.

## Setup

We use [torchhd](https://github.com/hyperdimensional-computing/torchhd), an open source library
for HDC built on top of PyTorch, and call its functions directly so you see the real API.
`sales_orders.py` holds the `NODES` and `EDGES` from notebook 1, so we don't repeat them here.
Import it before `lancedb`, because it also hides LanceDB's first-write log line.

In [1]:
from sales_orders import NODES, EDGES  # sales_orders.py sits next to this notebook

import lancedb
import polars as pl
import torch
import torchhd

## Defining a hypervector

What we want: one hypervector for every node and every relationship type in the graph. That
table is the **codebook**.

How: each hypervector has $D = 10{,}000$ entries, each $+1$ or $-1$ with equal chance.
torchhd calls this the **MAP** model, after its three operations: Multiply, Add, Permute. We
fix the random seed, so every run (and every later notebook) gets the same hypervectors.

![Illustrative hypervector strip with 10,000 dimensions and positive and negative entries](../img/hdc-hypervector.png)

*From slide 4: an illustrative excerpt of a hypervector. Here every node and relationship type gets 10,000 entries, each +1 or −1.*

In [2]:
torch.manual_seed(0)
D = 10_000

names = [n["name"] for rows in NODES.values() for n in rows] + list(EDGES)
hv = torchhd.random(len(names), D, vsa="MAP")
codebook = dict(zip(names, hv))  # sales_orders.make_codebook() does exactly this, for notebooks 3 and 4

print(len(names), "hypervectors")
print("first 10 entries of east-london:", codebook["east-london"][:10].tolist())

58 hypervectors
first 10 entries of east-london: [1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0]


## How alike are two hypervectors?

We compare hypervectors with **cosine similarity**:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|} = \frac{1}{D} \sum_{i=1}^{D} a_i b_i \quad \text{(for } \pm 1 \text{ entries)}$$

It's $1$ for identical hypervectors, $0$ for unrelated ones and $-1$ for exact opposites.

## Random hypervectors are almost orthogonal

This is the key fact behind HDC: two random hypervectors are almost unrelated. Each product
$a_i b_i$ is $+1$ or $-1$ at random, so the sum mostly cancels out. Its typical size is
$\sqrt{D}$, which means

$$\cos(\mathbf{a}, \mathbf{b}) \approx 0 \pm \frac{1}{\sqrt{D}} = 0 \pm 0.01$$

So every node gets its own, clearly distinct hypervector, without anyone designing it. Even
`east-london` and `west-london`, which sound alike, are as unrelated as any other pair.

In [3]:
for a, b in [("east-london", "west-london"), ("ireland", "uk"), ("lamps", "sofas"), ("SHIPS_FROM", "SHIPS_TO")]:
    print(f"cos({a}, {b}) = {torchhd.cosine_similarity(codebook[a], codebook[b]).item():+.3f}")

cos(east-london, west-london) = +0.018
cos(ireland, uk) = +0.004
cos(lamps, sofas) = +0.003
cos(SHIPS_FROM, SHIPS_TO) = +0.003


As seen above, the cosine similarity scores are hovering more or less around ~0.01 or that order of magnitude (near-orthogonal).

In [4]:
sims = torchhd.cosine_similarity(hv, hv)
off_diagonal = sims[~torch.eye(len(names), dtype=torch.bool)]
print(f"largest |cos| between two different hypervectors: {off_diagonal.abs().max().item():.3f}")
assert off_diagonal.abs().max() < 5 / D**0.5

largest |cos| between two different hypervectors: 0.038


## Store the codebook in LanceDB

Why we need a codebook at all: HDC computes only with hypervectors, so every name in the graph
must become one before we can combine them. The codebook is that mapping, and it has to work
in both directions:

- **Name → hypervector, always the same one.** Two orders that ship to Ireland both contain
  the *same* `ireland` hypervector, and that shared hypervector is what makes them similar
  later. A fresh random vector each time would make every order look unrelated.
- **Hypervector → name.** After we combine hypervectors, the result is just 10,000 numbers. To
  read the answer, we find the codebook entry closest to it (that's clean-up, in the next
  section).

The second direction is a nearest-neighbour search, which is what LanceDB does. So we store the
codebook there, with a `name`, a `kind` (the node label, or `relationship`) and the
hypervector itself as a fixed-size list of 10,000 floats.

In [5]:
kinds = {n["name"]: label for label, rows in NODES.items() for n in rows} | {r: "relationship" for r in EDGES}

db = lancedb.connect("sales.lancedb")
table = db.create_table(
    "codebook",
    [{"name": n, "kind": kinds[n], "vector": codebook[n].tolist()} for n in names],
    mode="overwrite",
)
table.schema

name: string
kind: string
vector: fixed_size_list<item: float>[10000]
  child 0, item: float

Here's what a few rows look like. Every row is one symbol from the graph, with its own column
of random $\pm 1$ entries. We only print the first eight of the 10,000.

In [6]:
preview = (
    table.search()
    .where("name IN ('SO-1042', 'Liffey Interiors', 'east-london', 'ireland', 'lamps', 'SHIPS_FROM')")
    .select(["name", "kind", "vector"])
    .to_polars()
)
pl.Config.set_fmt_table_cell_list_len(8)  # print all 8 entries, not the first and last
preview.with_columns(pl.col("vector").arr.head(8).cast(pl.List(pl.Int8)))

name,kind,vector
str,str,list[i8]
"""Liffey Interiors""","""Customer""","[1, 1, -1, 1, 1, 1, -1, -1]"
"""SO-1042""","""Order""","[-1, 1, -1, 1, -1, 1, 1, 1]"
"""lamps""","""Category""","[-1, 1, 1, -1, 1, -1, -1, 1]"
"""east-london""","""Depot""","[1, -1, -1, 1, -1, -1, -1, 1]"
"""ireland""","""Country""","[-1, 1, -1, 1, -1, 1, 1, -1]"
"""SHIPS_FROM""","""relationship""","[1, 1, -1, -1, 1, 1, 1, 1]"


## Clean-up: from a noisy hypervector back to a name

Why we need this: the operators below produce **noisy** hypervectors, close to a codebook
entry but not equal to it. To read the answer, we have to name the closest entry.

How: LanceDB's vector search finds the nearest codebook entries by cosine. LanceDB reports
a cosine *distance*, $1 - \cos$, so we convert it back to a similarity score.

In [7]:
def cleanup(query, k=3):
    hits = table.search(query.numpy()).metric("cosine").limit(k).to_list()
    return [(h["name"], round(1 - h["_distance"], 3)) for h in hits]

To test it, flip a random 30% of the entries of `ireland`. That's a lot of damage, but the
damaged copy is still far closer to `ireland` than to anything else in the codebook. Flipping a
fraction $p$ of the entries gives a cosine of exactly $1 - 2p$, here $0.4$, while every other
entry sits near $0$.

In [8]:
# Create a noisy version of the hypervector for "ireland"
noisy = codebook["ireland"].clone()

# Deliberately damage the hypervector by flipping 30% of its bits
noisy[torch.randperm(D)[: int(0.3 * D)]] *= -1
# Cleanup to recover the most similar hypervector from the codebook
cleanup(noisy)[0]

('ireland', 0.4)

## Binding ⊗

What it's for: storing a **key-value pair**, such as "ships from: East London", as one
hypervector. In a graph, the key is the relationship type and the value is the node at the
other end.

How: binding multiplies two hypervectors element by element:

$$(\mathbf{a} \otimes \mathbf{b})_i = a_i \, b_i$$

The result is again a ±1 hypervector of length $D$, and it's **unlike both inputs**:
$\cos(\mathbf{a} \otimes \mathbf{b}, \mathbf{a}) \approx 0$.

![Binding an obstacle role and a pallet value into a fact hypervector](../img/hdc-binding.png)

*From slide 5: the robot binds `obstacle ⊗ pallet`; our order binds `SHIPS_FROM ⊗ east-london`. The coloured patterns illustrate the operation, rather than its numerical values.*

In [9]:
pair = torchhd.bind(codebook["SHIPS_FROM"], codebook["east-london"])
print(f"cos(SHIPS_FROM ⊗ east-london, SHIPS_FROM)  = {torchhd.cosine_similarity(pair, codebook['SHIPS_FROM']).item():+.3f}")
print(f"cos(SHIPS_FROM ⊗ east-london, east-london) = {torchhd.cosine_similarity(pair, codebook['east-london']).item():+.3f}")

cos(SHIPS_FROM ⊗ east-london, SHIPS_FROM)  = -0.013
cos(SHIPS_FROM ⊗ east-london, east-london) = -0.004


### Binding reverses itself

To read the value back, bind with the key again. For ±1 entries, $a_i \cdot a_i = 1$, so
$\mathbf{a} \otimes \mathbf{a} = \mathbf{1}$, the all-ones hypervector, and the key cancels
out:

$$\mathbf{a} \otimes (\mathbf{a} \otimes \mathbf{b}) = (\mathbf{a} \otimes \mathbf{a}) \otimes \mathbf{b} = \mathbf{b}$$

In MAP, unbinding is the same operation as binding.

In [10]:
value = torchhd.bind(codebook["SHIPS_FROM"], pair)

print(cleanup(value))
assert torch.equal(value, codebook["east-london"])

[('east-london', 1.0), ('CONTAINS', 0.032), ('hauts-de-france', 0.028)]


## Bundling ⊕

What it's for: putting several things into **one** hypervector, like a set, while each of
them can still be found.

How: bundling adds hypervectors element by element:

$$(\mathbf{a} \oplus \mathbf{b} \oplus \mathbf{c})_i = a_i + b_i + c_i$$

Where binding gives a result unlike its inputs, bundling gives a result **like all of its
inputs**. Each input contributes $D$ to the dot product, and the rest is near-zero noise.
The sum's Euclidean magnitude is about $\sqrt{3D}$, so

$$\cos(\mathbf{a} \oplus \mathbf{b} \oplus \mathbf{c},\ \mathbf{a}) \approx \frac{D}{\sqrt{D}\,\sqrt{3D}} = \frac{1}{\sqrt{3}} \approx 0.577$$

Importantly, the number ~0.58 above shows that the similarity between a bundled item and any one of its components **scores clearly above unrelated items** (whose similarities are near zero).

![Bundling three bound robot facts into an episode hypervector](../img/hdc-bundling.png)

*From slide 5: bundling combines robot facts into an episode. We use the same addition to combine order facts into a record. The vector patterns are illustrative.*

In [11]:
depots = torchhd.multiset(torch.stack([codebook[n] for n in ["east-london", "west-london", "midlands"]]))

for n in ["east-london", "west-london", "midlands", "ireland", "lamps"]:
    print(f"cos(east-london ⊕ west-london ⊕ midlands, {n:11}) = {torchhd.cosine_similarity(depots, codebook[n]).item():+.3f}")

cos(east-london ⊕ west-london ⊕ midlands, east-london) = +0.578
cos(east-london ⊕ west-london ⊕ midlands, west-london) = +0.593
cos(east-london ⊕ west-london ⊕ midlands, midlands   ) = +0.580
cos(east-london ⊕ west-london ⊕ midlands, ireland    ) = +0.017
cos(east-london ⊕ west-london ⊕ midlands, lamps      ) = +0.002


### A sales order record as one hypervector

Let's now combine binding and bundling. **SO-1042 is the ID of a sales order record.**
We'll represent that order using three facts: it contains the product `cordless-table-lamp`,
ships from the depot `east-london`, and ships to `dublin`. We bind each relationship type
to its value and bundle the three pairs into one order-record hypervector:

$$\mathbf{SO1042_{rec}} = \mathbf{CONTAINS} \otimes \mathbf{cordless\text{-}table\text{-}lamp} \ \oplus\ \mathbf{SHIPS\_FROM} \otimes \mathbf{east\text{-}london} \ \oplus\ \mathbf{SHIPS\_TO} \otimes \mathbf{dublin}$$

In [12]:
facts = [("CONTAINS", "cordless-table-lamp"), ("SHIPS_FROM", "east-london"), ("SHIPS_TO", "dublin")]

# Bundle the three bound order facts by summing them element by element.
record = torchhd.multiset(torch.stack([torchhd.bind(codebook[k], codebook[v]) for k, v in facts]))
record[:8]

MAPTensor([ 1., -1.,  1., -1., -1., -1., -1.,  1.])

### Asking the record a question

"Where did SO-1042 ship from?" Binding distributes over bundling, so binding the whole record
with `SHIPS_FROM` unbinds that one pair and turns the other two into noise:

$$\mathbf{SHIPS\_FROM} \otimes \mathbf{SO1042_{rec}} = \mathbf{east\text{-}london} \oplus \text{noise}$$

The answer scores about $1/\sqrt{3} \approx 0.58$, and clean-up recovers it from the source data.

In [13]:
answer = torchhd.bind(codebook["SHIPS_FROM"], record)
hits = cleanup(answer)
print(hits)
assert hits[0][0] == "east-london" and hits[0][1] > 0.5

[('east-london', 0.579), ('west-london', 0.023), ('CONTAINS', 0.018)]


In [14]:
cleanup(torchhd.bind(codebook["SHIPS_TO"], record))

[('dublin', 0.583), ('midlands', 0.019), ('desks', 0.018)]

### Bundles have a capacity

Every item you bundle in lowers each member's score, as $1/\sqrt{k}$ for $k$ items. The
noise from non-members stays at about $1/\sqrt{D}$, so members stay easy to find until $k$
gets large. A bigger $D$ buys you room for more items.

In [15]:
for k in [1, 2, 4, 8, 16, len(names) - 1]:
    bundle = torchhd.multiset(hv[:k])
    member = torchhd.cosine_similarity(bundle, hv[0]).item()
    outsider = torchhd.cosine_similarity(bundle, hv[-1]).item()
    print(f"k = {k:2}   member {member:.3f}   1/√k = {k**-0.5:.3f}   non-member {outsider:+.3f}")

k =  1   member 1.000   1/√k = 1.000   non-member +0.007
k =  2   member 0.709   1/√k = 0.707   non-member +0.006
k =  4   member 0.492   1/√k = 0.500   non-member +0.004
k =  8   member 0.360   1/√k = 0.354   non-member +0.006
k = 16   member 0.256   1/√k = 0.250   non-member +0.019
k = 57   member 0.120   1/√k = 0.132   non-member +0.009


## Permutation ρ

Say you have a scenario where the order of a sequence matters. **Which depot was the shipment scanned at last?** East London → West London and
West London → East London need different answers. But plain bundling loses that order,
even when each scan is a record of bound field–value pairs: $a \oplus b = b \oplus a$.

Permutation can capture that order $\rho$ by shifting a vector's elements cyclically. We shift the elements of the **second** scan's hypervector by one position each **before** bundling to mark its position:

$$s_{AB} = a \oplus \rho(b), \qquad s_{BA} = b \oplus \rho(a)$$

Below, we represent each scan by just its depot hypervector. In torchhd, `shifts=1` marks the second position;
`shifts=-1` undoes that shift.


In [16]:
# Represent the two scans by their depot hypervectors.
a, b = codebook["east-london"], codebook["west-london"]
# Plain bundling loses order: both histories give the same vector.
print("Same without position:", torch.equal(a + b, b + a))
# Shift the second scan to distinguish East → West from West → East.
ab = a + torchhd.permute(b, shifts=1)
ba = b + torchhd.permute(a, shifts=1)
print("Same with position:", torch.equal(ab, ba))


Same without position: True
Same with position: False


In notebook 3, we use the same idea for graph paths: $\rho$ marks the first hop,
$\rho^2$ the second. Binding connects each relationship to its position, then bundling
combines the paths into one order representation.


## Wrap-up

| Operator | Math | What it does to similarity | Reversible? |
|---|---|---|---|
| binding ⊗ | $a_i \, b_i$ | result is **unlike** its inputs | yes: bind again with the same key |
| bundling ⊕ | $a_i + b_i$ | result is **like all** of its inputs, about $1/\sqrt{k}$ each | approximately: clean up against the codebook |
| permutation ρ | cyclic shift | result is **unlike** its input | yes: $\rho^{-1}$ |

Every result is still one hypervector of length $D$. In notebook 3 we use all three to turn
an order's pattern into one hypervector. Each path out of the order becomes a key-value pair:
the relationship types along the path, with ρ marking each hop's position, bound to the node
at the end. For example, "SO-1042 ships to a region in Ireland" becomes

$$\mathbf{SHIPS\_TO} \otimes \rho(\mathbf{IN\_COUNTRY}) \otimes \rho^2(\mathbf{ireland})$$

and bundling an order's paths gives its **motif hypervector**, which we can search for.